# Canadian Rental Market Pressure Analysis

This notebook cleans and prepares the CMHC Table 1.0 Rental Market Survey data from 2024 and 2025. It handles missing values, filters out aggregate provincial and national rows, and saves the final dataset to a local CSV file and an SQLite database so we can easily query it in the next step.

In [1]:
import os
import sqlite3
import pandas as pd
import numpy as np

# Ensure output directories exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../database', exist_ok=True)
os.makedirs('../outputs/tables', exist_ok=True)

## Load raw data and map headers

The CMHC Excel sheet has multi-row headers. I load it without headers first, extract the main data section, and skip the footnotes at the bottom.

In [2]:
raw_file_path = '../data/raw/cmhc/cmhc_rental_market_survey_latest.xlsx'
sheet_name = 'Table 1.0'

# Load sheet without parsing headers to examine rows manually
df_raw = pd.read_excel(raw_file_path, sheet_name=sheet_name, header=None)
print(f"Raw sheet shape: {df_raw.shape}")

Raw sheet shape: (77, 19)


## Clean values and filter out aggregates

I filter out provincial and national aggregate rows to focus strictly on city-level data. Then I clean up commas and replace suppressed values (like  or ) with standard null values. I also calculate the change in vacancy rates and set helper flags for caution-quality records (estimate quality flag ) and valid metrics.

In [3]:
# 1. Extract data section (rows 7 to 61 inclusive)
df_data = df_raw.iloc[7:62].copy()

# Keep track of raw count and aggregates for metrics printout
total_data_section_rows = len(df_data)
is_aggregate = df_data[0].astype(str).str.contains('10,000\\+|Canada CMAs', na=True)
df_aggregates = df_data[is_aggregate]
df_cities = df_data[~is_aggregate].copy()

print(f"Rows in raw data section: {total_data_section_rows}")
print(f"Excluded aggregate rows count: {len(df_aggregates)}")
print(f"Initial city rows count: {len(df_cities)}")

# Helper function for converting numeric fields
def parse_numeric(val):
    if pd.isna(val):
        return np.nan
    val_str = str(val).strip()
    if val_str in ['**', '..', '++', '--', '-', 'nan', 'NaN', '']:
        return np.nan
    val_str = val_str.replace(',', '')
    try:
        return float(val_str)
    except ValueError:
        return np.nan

# Build target dataframe with standardized columns
cleaned_df = pd.DataFrame()

# Center name
cleaned_df['centre'] = df_cities[0].astype(str).str.strip()

# Rates and Rents
cleaned_df['vacancy_rate_2024'] = df_cities[1].apply(parse_numeric)
cleaned_df['vacancy_rate_2025'] = df_cities[3].apply(parse_numeric)
cleaned_df['turnover_rate_2024'] = df_cities[6].apply(parse_numeric)
cleaned_df['turnover_rate_2025'] = df_cities[8].apply(parse_numeric)
cleaned_df['average_rent_2br_2024'] = df_cities[11].apply(parse_numeric)
cleaned_df['average_rent_2br_2025'] = df_cities[13].apply(parse_numeric)
cleaned_df['rent_growth_pct_2024_2025'] = df_cities[17].apply(parse_numeric)

# Derived Fields
cleaned_df['vacancy_change_pct_points'] = (
    cleaned_df['vacancy_rate_2025'] - cleaned_df['vacancy_rate_2024']
).round(2)

# Quality Indicators for 2025
def parse_quality(val):
    if pd.isna(val):
        return None
    val_str = str(val).strip()
    if val_str in ['nan', 'NaN', 'None', '', '**', '++', '--', '-']:
        return None
    return val_str

cleaned_df['vacancy_quality_2025'] = df_cities[4].apply(parse_quality)
cleaned_df['rent_quality_2025'] = df_cities[14].apply(parse_quality)
cleaned_df['rent_growth_quality_2025'] = df_cities[18].apply(parse_quality)

# Validation/Filtering Boolean Flags
cleaned_df['has_valid_vacancy_2025'] = cleaned_df['vacancy_rate_2025'].notna()
cleaned_df['has_valid_rent_2025'] = cleaned_df['average_rent_2br_2025'].notna()
cleaned_df['has_valid_rent_growth'] = cleaned_df['rent_growth_pct_2024_2025'].notna()

# is_caution_quality is True if any of the three 2025 quality indicators is 'd'
cleaned_df['is_caution_quality'] = (
    (cleaned_df['vacancy_quality_2025'] == 'd') |
    (cleaned_df['rent_quality_2025'] == 'd') |
    (cleaned_df['rent_growth_quality_2025'] == 'd')
).fillna(False)

print(f"Processed cleaned dataframe shape: {cleaned_df.shape}")

Rows in raw data section: 55
Excluded aggregate rows count: 12
Initial city rows count: 43
Processed cleaned dataframe shape: (43, 16)


## Data quality checks

A quick check to make sure city names are unique, not empty, and that rates are within valid percentage bounds.

In [4]:
# Validation checks
errors = []

# 1. Missing Centre Names
missing_names = cleaned_df['centre'].isna() | (cleaned_df['centre'] == '')
if missing_names.any():
    errors.append(f"CRITICAL: Found {missing_names.sum()} rows with missing centre names!")

# 2. Duplicate Centre names
duplicates = cleaned_df['centre'].duplicated()
if duplicates.any():
    errors.append(f"CRITICAL: Found duplicate centre rows: {cleaned_df.loc[duplicates, 'centre'].tolist()}!")

# 3. Out-of-bounds rates
invalid_vacancy = (
    (cleaned_df['vacancy_rate_2024'] < 0) | (cleaned_df['vacancy_rate_2024'] > 100) |
    (cleaned_df['vacancy_rate_2025'] < 0) | (cleaned_df['vacancy_rate_2025'] > 100)
).fillna(False)
if invalid_vacancy.any():
    errors.append(f"CRITICAL: Found out-of-bounds vacancy rates in: {cleaned_df.loc[invalid_vacancy, 'centre'].tolist()}!")

invalid_turnover = (
    (cleaned_df['turnover_rate_2024'] < 0) | (cleaned_df['turnover_rate_2024'] > 100) |
    (cleaned_df['turnover_rate_2025'] < 0) | (cleaned_df['turnover_rate_2025'] > 100)
).fillna(False)
if invalid_turnover.any():
    errors.append(f"CRITICAL: Found out-of-bounds turnover rates in: {cleaned_df.loc[invalid_turnover, 'centre'].tolist()}!")

# 4. Negative rents
negative_rents = (
    (cleaned_df['average_rent_2br_2024'] < 0) |
    (cleaned_df['average_rent_2br_2025'] < 0)
).fillna(False)
if negative_rents.any():
    errors.append(f"CRITICAL: Found negative rents in: {cleaned_df.loc[negative_rents, 'centre'].tolist()}!")

# Report findings
if errors:
    for err in errors:
        print(err)
    raise AssertionError("Validation checks failed! Refer to messages above.")
else:
    print("SUCCESS: All validations passed! No duplicates, negative values, or out-of-bounds rates found.")

SUCCESS: All validations passed! No duplicates, negative values, or out-of-bounds rates found.


## Export cleaned data

I save the cleaned dataset to a CSV file and load it into a local SQLite table.

In [5]:
# 1. Save Cleaned DataFrame to CSV
cleaned_csv_path = "../data/processed/cmhc_rental_market_2024_2025_cleaned.csv"
cleaned_df.to_csv(cleaned_csv_path, index=False)
print(f"Saved cleaned data to: {cleaned_csv_path}")

# 2. Save to SQLite Database
db_path = "../database/rental_market.db"
conn = sqlite3.connect(db_path)
try:
    # Using if_exists="replace" to recreate the table cleanly
    cleaned_df.to_sql("rental_market_2024_2025", conn, if_exists="replace", index=False)
    print(f"Successfully wrote table 'rental_market_2024_2025' to SQLite database: {db_path}")
finally:
    conn.close()


Saved cleaned data to: ../data/processed/cmhc_rental_market_2024_2025_cleaned.csv
Successfully wrote table 'rental_market_2024_2025' to SQLite database: ../database/rental_market.db


## Summary of results

Let's check the final counts and see a preview of the clean data.

In [6]:
raw_count_total = df_raw.shape[0]
final_city_count = len(cleaned_df)
valid_rent_2025 = cleaned_df['has_valid_rent_2025'].sum()
valid_vacancy_2025 = cleaned_df['has_valid_vacancy_2025'].sum()
caution_quality_count = cleaned_df['is_caution_quality'].sum()
excluded_aggregate_count = len(df_aggregates)

print("=================== PIPELINE EXECUTION SUMMARY ===================")
print(f"Raw sheet row count:             {raw_count_total}")
print(f"Final city-level row count:       {final_city_count}")
print(f"Number of valid 2025 rent values: {valid_rent_2025}")
print(f"Number of valid 2025 vacancy:     {valid_vacancy_2025}")
print(f"Number of records flagged caution:{caution_quality_count}")
print(f"Number of excluded aggregate rows: {excluded_aggregate_count}")
print("==================================================================")

print("\nFirst 15 cleaned rows:")
print(cleaned_df.head(15).to_string())

=================== PIPELINE EXECUTION SUMMARY ===================
Raw sheet row count:             77
Final city-level row count:       43
Number of valid 2025 rent values: 43
Number of valid 2025 vacancy:     43
Number of records flagged caution:5
Number of excluded aggregate rows: 12

First 15 cleaned rows:
                             centre  vacancy_rate_2024  vacancy_rate_2025  turnover_rate_2024  turnover_rate_2025  average_rent_2br_2024  average_rent_2br_2025  rent_growth_pct_2024_2025  vacancy_change_pct_points vacancy_quality_2025 rent_quality_2025 rent_growth_quality_2025  has_valid_vacancy_2025  has_valid_rent_2025  has_valid_rent_growth  is_caution_quality
8                    St. John's CMA                2.1                2.0                14.1                13.0                 1250.0                 1361.0                        7.6                       -0.1                    a                 b                        b                    True                 True